# 🏦 Notebook 3 — End-to-End Worked Example: Strangling a Banking Monolith

We'll migrate a small legacy banking app to a new service, slice by slice, using
everything from Notebooks 1 and 2 plus two new ideas:

- **Dual-write** for data migration (so both systems see new data during cutover).
- **Per-endpoint progress tracking** with a tiny metrics dashboard.

All in pure Python — no servers, no DB. Just dictionaries pretending to be a bank.

## 🛠️ Setup

```bash
cd 05-microservices/strangler
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

> This lab uses **only the Python standard library** — no servers, no databases. Every cell is a small simulation you can read end-to-end.


## 1. The legacy bank (don't touch it!)

Four operations: `create_account`, `deposit`, `withdraw`, `balance`.
Data lives in a global dict (pretend it's an ancient Oracle DB).

In [1]:
LEGACY_DB = {}   # account_id -> balance in cents

class LegacyBank:
    name = "legacy-bank-v1"
    def create_account(self, account_id):
        LEGACY_DB.setdefault(account_id, 0)
        return {"ok": True, "by": self.name}
    def deposit(self, account_id, cents):
        LEGACY_DB[account_id] = LEGACY_DB.get(account_id, 0) + cents
        return {"ok": True, "by": self.name, "balance": LEGACY_DB[account_id]}
    def withdraw(self, account_id, cents):
        if LEGACY_DB.get(account_id, 0) < cents:
            return {"ok": False, "by": self.name, "error": "insufficient_funds"}
        LEGACY_DB[account_id] -= cents
        return {"ok": True, "by": self.name, "balance": LEGACY_DB[account_id]}
    def balance(self, account_id):
        return {"ok": True, "by": self.name, "balance": LEGACY_DB.get(account_id, 0)}


## 2. The new bank (clean rewrite, built one slice at a time)

In [2]:
NEW_DB = {}

class NewBank:
    name = "new-bank-v2"
    def create_account(self, account_id):
        if account_id in NEW_DB:
            return {"ok": True, "by": self.name, "note": "already_exists"}
        NEW_DB[account_id] = 0
        return {"ok": True, "by": self.name}
    def deposit(self, account_id, cents):
        NEW_DB[account_id] = NEW_DB.get(account_id, 0) + cents
        return {"ok": True, "by": self.name, "balance": NEW_DB[account_id]}
    def withdraw(self, account_id, cents):
        if NEW_DB.get(account_id, 0) < cents:
            return {"ok": False, "by": self.name, "error": "insufficient_funds"}
        NEW_DB[account_id] -= cents
        return {"ok": True, "by": self.name, "balance": NEW_DB[account_id]}
    def balance(self, account_id):
        return {"ok": True, "by": self.name, "balance": NEW_DB.get(account_id, 0)}


## 3. The facade — a richer router

This facade tracks, per operation:
- **% traffic** on the new service,
- whether to **dual-write** (write to both DBs during migration),
- a simple **metrics counter** so we can see what's happening.


In [3]:
import random, collections, time

class BankingFacade:
    def __init__(self, legacy, new):
        self.legacy = legacy
        self.new = new
        # op -> {"pct_new": 0..100, "dual_write": bool}
        self.plan = collections.defaultdict(lambda: {"pct_new": 0, "dual_write": False})
        self.metrics = collections.Counter()   # (op, target) -> count
        self.latency_ms = collections.defaultdict(list)

    def configure(self, op, pct_new=None, dual_write=None):
        if pct_new is not None:   self.plan[op]["pct_new"] = pct_new
        if dual_write is not None: self.plan[op]["dual_write"] = dual_write

    def _call(self, target, op, *args):
        t0 = time.perf_counter()
        result = getattr(target, op)(*args)
        self.latency_ms[(op, target.name)].append((time.perf_counter() - t0) * 1000)
        self.metrics[(op, target.name)] += 1
        return result

    def handle(self, op, *args):
        cfg = self.plan[op]
        use_new = random.randint(1, 100) <= cfg["pct_new"]
        primary = self.new if use_new else self.legacy
        secondary = self.legacy if use_new else self.new

        result = self._call(primary, op, *args)

        # Dual-write: mirror the write to the other system so its data stays fresh.
        # Only for write ops — never dual-write reads.
        if cfg["dual_write"] and op in {"create_account", "deposit", "withdraw"}:
            try:
                self._call(secondary, op, *args)
            except Exception as e:
                # Mirror failure should never break the primary response.
                self.metrics[("mirror_error", secondary.name)] += 1
        return result

    def report(self):
        print("--- traffic counts ---")
        for (op, target), count in sorted(self.metrics.items()):
            print(f"  {op:16s} {target:16s} {count:5d}")


## 4. The migration in 5 phases

We simulate a workload at each phase and watch the routing shift.

In [4]:
def simulate(facade, n=500):
    """Pretend customers are doing stuff."""
    for _ in range(n):
        aid = random.choice(["A1", "A2", "A3", "A4"])
        op = random.choices(
            ["create_account", "deposit", "withdraw", "balance"],
            weights=[1, 5, 3, 10],
        )[0]
        if op == "create_account":
            facade.handle(op, aid)
        elif op == "balance":
            facade.handle(op, aid)
        else:
            facade.handle(op, aid, random.randint(100, 5000))

random.seed(42)
LEGACY_DB.clear(); NEW_DB.clear()
facade = BankingFacade(LegacyBank(), NewBank())

print("Phase 1 — facade in place, 100% still on legacy (no behavioural change).")
simulate(facade, 300)
facade.report()


Phase 1 — facade in place, 100% still on legacy (no behavioural change).
--- traffic counts ---
  balance          legacy-bank-v1     158
  create_account   legacy-bank-v1      13
  deposit          legacy-bank-v1      78
  withdraw         legacy-bank-v1      51


In [5]:
print("\nPhase 2 — canary: 10% of `balance` (read-only) to the new service.")
facade.configure("balance", pct_new=10)
simulate(facade, 300)
facade.report()



Phase 2 — canary: 10% of `balance` (read-only) to the new service.
--- traffic counts ---
  balance          legacy-bank-v1     314
  balance          new-bank-v2         11
  create_account   legacy-bank-v1      29
  deposit          legacy-bank-v1     155
  withdraw         legacy-bank-v1      91


In [6]:
print("\nPhase 3 — ramp balance to 100%; start DUAL-WRITING deposits so NEW_DB catches up.")
facade.configure("balance", pct_new=100)
facade.configure("deposit", pct_new=0, dual_write=True)   # still read/write legacy, but mirror writes
simulate(facade, 300)
facade.report()
print("\nLEGACY_DB:", LEGACY_DB)
print("NEW_DB   :", NEW_DB)



Phase 3 — ramp balance to 100%; start DUAL-WRITING deposits so NEW_DB catches up.
--- traffic counts ---
  balance          legacy-bank-v1     314
  balance          new-bank-v2        171
  create_account   legacy-bank-v1      44
  deposit          legacy-bank-v1     229
  deposit          new-bank-v2         74
  withdraw         legacy-bank-v1     142

LEGACY_DB: {'A1': 62531, 'A2': 42600, 'A4': 38893, 'A3': 54297}
NEW_DB   : {'A3': 50398, 'A4': 51550, 'A1': 49761, 'A2': 33219}


In [7]:
print("\nPhase 4 — now that data is mirrored, flip deposits and withdraws to the new service.")
facade.configure("deposit",  pct_new=100, dual_write=True)   # keep dual-write the other way for safety
facade.configure("withdraw", pct_new=100, dual_write=True)
facade.configure("create_account", pct_new=100, dual_write=True)
simulate(facade, 300)
facade.report()



Phase 4 — now that data is mirrored, flip deposits and withdraws to the new service.
--- traffic counts ---
  balance          legacy-bank-v1     314
  balance          new-bank-v2        326
  create_account   legacy-bank-v1      63
  create_account   new-bank-v2         19
  deposit          legacy-bank-v1     313
  deposit          new-bank-v2        158
  withdraw         legacy-bank-v1     184
  withdraw         new-bank-v2         42


In [8]:
print("\nPhase 5 — retirement: confident, turn dual-write off. Legacy stops receiving traffic.")
for op in ["create_account", "deposit", "withdraw", "balance"]:
    facade.configure(op, pct_new=100, dual_write=False)

# Reset counters so this phase's report shows ONLY the phase-5 traffic.
facade.metrics.clear()
facade.latency_ms.clear()

simulate(facade, 300)
facade.report()
print("\nNotice: no `legacy-bank-v1` rows. Legacy received 0 calls in phase 5 — safe to delete.")



Phase 5 — retirement: confident, turn dual-write off. Legacy stops receiving traffic.
--- traffic counts ---
  balance          new-bank-v2        174
  create_account   new-bank-v2         11
  deposit          new-bank-v2         66
  withdraw         new-bank-v2         49

Notice: no `legacy-bank-v1` rows. Legacy received 0 calls in phase 5 — safe to delete.


## 5. Looking at the latency log

In [9]:
def p(name, vals):
    if not vals: return
    vals = sorted(vals)
    p50 = vals[len(vals)//2]
    p95 = vals[int(len(vals)*0.95)]
    print(f"  {name:40s} n={len(vals):4d}  p50={p50:.3f}ms  p95={p95:.3f}ms")

print("Latency by (operation, target):")
for key, vals in sorted(facade.latency_ms.items()):
    p(f"{key[0]} @ {key[1]}", vals)


Latency by (operation, target):
  balance @ new-bank-v2                    n= 174  p50=0.000ms  p95=0.000ms
  create_account @ new-bank-v2             n=  11  p50=0.000ms  p95=0.001ms
  deposit @ new-bank-v2                    n=  66  p50=0.000ms  p95=0.001ms
  withdraw @ new-bank-v2                   n=  49  p50=0.000ms  p95=0.001ms


## 6. Lessons from this worked example

1. **Phase 1 is a no-op deploy** — putting the facade in place with 0% migration
   is its own scary deploy. Do it on its own.
2. **Reads before writes.** `balance` (read-only) was migrated first. Bugs in
   reads are annoying; bugs in writes corrupt data.
3. **Dual-write bridges the data gap.** Before flipping writes to the new DB,
   we mirror writes to it for a while so it has real data to work with. After
   cutover we keep mirroring *back* to legacy for rollback safety, then finally
   turn it off.
4. **The facade owns the migration state.** Nothing in legacy or new needs to
   know a migration is happening. That's the whole point.
5. **Retirement is a deliberate step.** Don't let legacy linger "just in case"
   forever.


## 7. 🌍 Variations you'll see in the wild

- **Event interception** — instead of HTTP routing, intercept at the *event bus*
  (e.g. Kafka). The new service subscribes to the same events; when ready, the
  old consumer is turned off. Used heavily in CQRS / event-sourced systems.
- **Branch by abstraction** — introduce an interface inside the legacy codebase
  and slowly reimplement it. Good when you *can* modify legacy code but want
  to avoid a big-bang branch.
- **Change data capture (CDC)** — instead of dual-writing from the app, use a
  tool like Debezium to stream every legacy DB change into the new DB. The app
  stays unchanged; the DB is strangled.
- **Strangler at the UI layer** — mount the new app on a sub-path (`/v2/*`) and
  migrate pages one at a time. LinkedIn and Etsy have written about this.

All of them are the same idea: **one routing layer, one tiny slice at a time,
always reversible.**


## 8. 📚 Further reading

- Martin Fowler — *StranglerFigApplication* (2004) — the original essay.
- Sam Newman — *Monolith to Microservices* (O'Reilly, 2019) — the canonical book.
- GitHub's *Scientist* library — [github/scientist](https://github.com/github/scientist) — production dark-launch tool.
- Shopify engineering blog — "Deconstructing the Monolith".
- Stripe engineering — "Online migrations at scale".

### 🎯 You now know
- Why big-bang rewrites fail, and how the strangler fig avoids that.
- The three pieces: legacy, new, facade.
- Canary routing, dark launch, kill switches.
- Dual-write for data migration.
- The five phases of a real migration — including retirement.
